In [2]:
import sagemaker
from sagemaker.inputs import FileSystemInput
from sagemaker.tensorflow import TensorFlow
from sagemaker.debugger import TensorBoardOutputConfig

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Docker Image Details
ACCOUNT_ID = "211125578777"
REGION = "us-east-1"
DOCKER_REPOSITORY = "streamswitch-docker-images"

# FSx details
FILE_SYSTEM_ID = "fs-07f6c36209d77bd87"
MOUNT_POINT = "/wjgmlamv" 
ACCESS_MODE = "ro"
FILE_SYSTEM_TYPE = "FSxLustre"

In [9]:
custom_image_uri = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{DOCKER_REPOSITORY}:gpu-optimized"
session = sagemaker.Session()

# Define the FileSystemInput
fsx_input = FileSystemInput(
    file_system_id=FILE_SYSTEM_ID,
    file_system_type=FILE_SYSTEM_TYPE,
    directory_path=MOUNT_POINT,
    file_system_access_mode=ACCESS_MODE
)

tensorboard_output_config = TensorBoardOutputConfig(
    s3_output_path='s3://streamswitch-training-results',
    container_local_output_path='/opt/ml/output/tensorboard'
)

In [ ]:
estimator = TensorFlow(
    sagemaker_session=session,
    entry_point='streamswitch.py',
    source_dir='.',
    base_job_name="streamswitch-training-job",
    image_uri=custom_image_uri,
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type="ml.g5.12xlarge",
    subnets=['subnet-099c27aa0c4534730', 'subnet-0345aeff59a7d11fd', 'subnet-02d2c1ee7d1232b97', 'subnet-039b7ea4dd5e37d97', 'subnet-0ec70f3e1d5eb7349', 'subnet-00a1cf5120d3ff292'],
    security_group_ids=['sg-01a1363df4a019c30'],
    tensorboard_output_config=tensorboard_output_config,
    run_tensorboard_locally=True,
    enable_remote_debug=True,
    environment={'PYTHONUNBUFFERED': '1', 'NCCL_DEBUG': 'INFO'},
    # checkpoint_s3_uri='s3://streamswitch-training-results/checkpoints',
    # checkpoint_local_path='/opt/ml/checkpoints'
    # output_path='s3://streamswitch-training-data',
)

estimator.fit({'train': fsx_input})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: streamswitch-training-job-2026-06-04-07-40-27-635


2026-06-04 07:40:36 Starting - Starting the training job
2026-06-04 07:40:36 Pending - Training job waiting for capacity.........
2026-06-04 07:41:53 Pending - Preparing the instances for training...